# Chapter 1 — Tensors & Shape Algebra (Practice)

Work through these exercises **after reading** `notes/ch01-tensors-and-shape-algebra.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Don't peek at `solutions/` until the verification passes or you're genuinely stuck.

In [1]:
# ============================================================
# TOPIC: Tensors & shape algebra — storage, strides, dtypes, broadcasting
# MATH:  position(i_0..i_{n-1}) = offset + sum_k i_k * stride_k
# REF:   B00 ch01 notes — tensors-and-shape-algebra
# ============================================================

# --- Imports ---
import numpy as np
import torch
import torch.nn as nn

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

torch version : 2.13.0
device        : cpu  (every exercise here runs fine on CPU)


## Exercise 1 — Token Batch Bootstrap

A tokenizer just handed you a **ragged** batch (sequences of different lengths). Build:

1. `padded` — an `int64` tensor of shape `(batch, max_len)`, short rows filled with `PAD_ID`
2. `pad_mask` — a `bool` tensor of the same shape, `True` where the token is real

**Decision you're practicing:** which dtype each artifact needs (`int64` for IDs because `nn.Embedding` is a table lookup; `bool` for masks) and which creation functions get you there (`torch.full`, `torch.zeros`, `dtype=`).

In [2]:
ragged_token_ids = [
    [5, 12, 7],
    [3, 9],
    [42, 8, 15, 2, 6],
]
PAD_ID = 0

batch_size = len(ragged_token_ids)
max_len = max(len(seq) for seq in ragged_token_ids)
print(f"batch_size = {batch_size}, max_len = {max_len}")

# torch.full pre-fills every slot with PAD_ID; dtype=torch.long because token IDs index an embedding table
padded = torch.full((batch_size, max_len), PAD_ID, dtype=torch.long)   # shape: (batch, max_len)
pad_mask = torch.zeros(batch_size, max_len, dtype=torch.bool)          # shape: (batch, max_len)

for row_index, sequence in enumerate(ragged_token_ids):
    seq_len = len(sequence)
    padded[row_index, :seq_len] = torch.tensor(sequence, dtype=torch.long)
    pad_mask[row_index, :seq_len] = True
    print(f"row {row_index}: {seq_len} real tokens copied, {max_len - seq_len} slots left as PAD_ID")

print(f"\npadded ({padded.dtype}, shape {tuple(padded.shape)}):\n{padded}")
print(f"\npad_mask ({pad_mask.dtype}, shape {tuple(pad_mask.shape)}):\n{pad_mask}")

batch_size = 3, max_len = 5
row 0: 3 real tokens copied, 2 slots left as PAD_ID
row 1: 2 real tokens copied, 3 slots left as PAD_ID
row 2: 5 real tokens copied, 0 slots left as PAD_ID

padded (torch.int64, shape (3, 5)):
tensor([[ 5, 12,  7,  0,  0],
        [ 3,  9,  0,  0,  0],
        [42,  8, 15,  2,  6]])

pad_mask (torch.bool, shape (3, 5)):
tensor([[ True,  True,  True, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True,  True,  True]])


**Verification** — run after filling the stub.

In [3]:
# --- Verification: Exercise 1 ---
assert padded is not None and pad_mask is not None, "fill in the stub above first"
assert padded.dtype == torch.long,   f"token IDs must be int64/long, got {padded.dtype}"
assert pad_mask.dtype == torch.bool, f"masks must be bool, got {pad_mask.dtype}"
assert padded.shape == (3, 5) and pad_mask.shape == (3, 5), "shape must be (batch=3, max_len=5)"
assert padded[0].tolist() == [5, 12, 7, 0, 0]
assert padded[1].tolist() == [3, 9, 0, 0, 0]
assert padded[2].tolist() == [42, 8, 15, 2, 6]
assert pad_mask.sum().item() == 3 + 2 + 5, "mask must be True at exactly the real-token positions"

toy_embedding = nn.Embedding(num_embeddings=100, embedding_dim=8, padding_idx=PAD_ID)
token_vectors = toy_embedding(padded)   # this line raises if padded is a float tensor
print(f"nn.Embedding accepted the batch → output shape {tuple(token_vectors.shape)}  # (batch, max_len, embed_dim)")
print("Exercise 1 passed ✓")

nn.Embedding accepted the batch → output shape (3, 5, 8)  # (batch, max_len, embed_dim)
Exercise 1 passed ✓


## Exercise 2 — Storage Detective

For each derived tensor below, **predict** whether it shares storage with `base`, then implement `shares_storage` to check yourself. (Hint from notes §2: views re-describe the same flat memory; fancy indexing and forced copies allocate new memory. `t.untyped_storage().data_ptr()` identifies the underlying block.)

**Decision you're practicing:** knowing *which operations alias memory* — the difference between a free view and a hidden copy.

In [4]:
base = torch.arange(12).view(3, 4)

derived = {
    "row_slice":            base[1],
    "transpose":            base.t(),
    "fancy_rows":           base[[2, 0]],
    "reshape_contiguous":   base.reshape(4, 3),
    "reshape_of_transpose": base.t().reshape(12),
    "contiguous_noop":      base.contiguous(),
}

predictions = {
    "row_slice":            True,    # basic slicing = view
    "transpose":            True,    # t() only swaps strides
    "fancy_rows":           False,   # index-list gather must copy
    "reshape_contiguous":   True,    # base is contiguous → reshape degrades to view
    "reshape_of_transpose": False,   # non-contiguous source → reshape must copy
    "contiguous_noop":      True,    # already contiguous → contiguous() returns self
}

def shares_storage(tensor_a, tensor_b):
    """Return True if the two tensors are backed by the same memory block."""
    return tensor_a.untyped_storage().data_ptr() == tensor_b.untyped_storage().data_ptr()

print(f"base storage id: {base.untyped_storage().data_ptr()}")
for name, tensor in derived.items():
    print(f"  {name:22s} shape {str(tuple(tensor.shape)):9s} shares={shares_storage(tensor, base)}")

base storage id: 4568078976
  row_slice              shape (4,)      shares=True
  transpose              shape (4, 3)    shares=True
  fancy_rows             shape (2, 4)    shares=False
  reshape_contiguous     shape (4, 3)    shares=True
  reshape_of_transpose   shape (12,)     shares=False
  contiguous_noop        shape (3, 4)    shares=True


**Verification**

In [5]:
# --- Verification: Exercise 2 ---
rules = {
    "row_slice":            "basic slicing is a view",
    "transpose":            "transpose only swaps strides",
    "fancy_rows":           "fancy indexing must gather scattered elements → copy",
    "reshape_contiguous":   "reshape on a contiguous tensor degrades to a view",
    "reshape_of_transpose": "reshape on non-contiguous memory must copy",
    "contiguous_noop":      "contiguous() on an already-contiguous tensor returns self",
}

def _shares(a, b):
    return a.untyped_storage().data_ptr() == b.untyped_storage().data_ptr()

wrong = 0
for name, tensor in derived.items():
    truth = _shares(tensor, base)
    assert predictions[name] is not None, f"no prediction for {name!r}"
    assert shares_storage(tensor, base) == truth, "your shares_storage() disagrees with data_ptr ground truth"
    mark = "✓" if predictions[name] == truth else "✗"
    wrong += predictions[name] != truth
    print(f"{mark} {name:22s} shares={str(truth):5s} — {rules[name]}")
assert wrong == 0, f"{wrong} prediction(s) wrong — revisit notes §2, §5, §6"
print("Exercise 2 passed ✓")

✓ row_slice              shares=True  — basic slicing is a view
✓ transpose              shares=True  — transpose only swaps strides
✓ fancy_rows             shares=False — fancy indexing must gather scattered elements → copy
✓ reshape_contiguous     shares=True  — reshape on a contiguous tensor degrades to a view
✓ reshape_of_transpose   shares=False — reshape on non-contiguous memory must copy
✓ contiguous_noop        shares=True  — contiguous() on an already-contiguous tensor returns self
Exercise 2 passed ✓


## Exercise 3 — Fix the `view()` Crash

The classic multi-head-attention merge bug. `context` is `(batch, heads, seq, head_dim)`; after `transpose(1, 2)` the tensor is **non-contiguous**, and this line explodes:

```python
context.transpose(1, 2).view(BATCH, SEQ_LEN, HEADS * HEAD_DIM)   # 💥 RuntimeError
```

Write **two** fixes and, in a comment, explain in stride terms why the original crashes:

- `merge_heads_explicit` — using `.contiguous().view(...)` (the copy is *visible*)
- `merge_heads_reshape` — using `.reshape(...)` (the copy happens *silently*)

**Decision you're practicing:** `view` (loud no-copy guarantee) vs `reshape` (silent maybe-copy) vs `.contiguous().view()` (explicit copy).

In [6]:
BATCH, HEADS, SEQ_LEN, HEAD_DIM = 2, 3, 4, 5
context = torch.randn(BATCH, HEADS, SEQ_LEN, HEAD_DIM)   # shape: (batch, heads, seq, head_dim)
print(f"context: shape {tuple(context.shape)}, stride {context.stride()}, contiguous={context.is_contiguous()}")

swapped = context.transpose(1, 2)                         # shape: (batch, seq, heads, head_dim) — a VIEW
print(f"swapped: shape {tuple(swapped.shape)}, stride {swapped.stride()}, contiguous={swapped.is_contiguous()}")

# Why the original crashes: view() must describe the merged (heads*head_dim) axis
# with ONE constant stride over the existing memory. But after transpose the
# strides are (60, 5, 20, 1): walking the heads axis jumps 20 slots while the
# head_dim axis jumps 1 — the merged axis would need to alternate jump sizes,
# which no single stride can express. contiguous() copies memory into the new
# row-major order, after which the merge is a trivial re-description.

def merge_heads_explicit(context_tensor):
    """(batch, heads, seq, head_dim) -> (batch, seq, heads*head_dim) via contiguous().view() — visible copy."""
    batch_size, num_heads, seq_len, head_dim = context_tensor.shape
    swapped_view = context_tensor.transpose(1, 2)          # (batch, seq, heads, head_dim), non-contiguous
    return swapped_view.contiguous().view(batch_size, seq_len, num_heads * head_dim)

def merge_heads_reshape(context_tensor):
    """Same merge via reshape() — also works, but the copy happens silently."""
    batch_size, num_heads, seq_len, head_dim = context_tensor.shape
    return context_tensor.transpose(1, 2).reshape(batch_size, seq_len, num_heads * head_dim)

print(f"explicit fix → shape {tuple(merge_heads_explicit(context).shape)}")
print(f"reshape  fix → shape {tuple(merge_heads_reshape(context).shape)}")

context: shape (2, 3, 4, 5), stride (60, 20, 5, 1), contiguous=True
swapped: shape (2, 4, 3, 5), stride (60, 5, 20, 1), contiguous=False
explicit fix → shape (2, 4, 15)
reshape  fix → shape (2, 4, 15)


**Verification**

In [7]:
# --- Verification: Exercise 3 ---
reference = context.permute(0, 2, 1, 3).contiguous().view(BATCH, SEQ_LEN, HEADS * HEAD_DIM)

crashed = False
try:
    context.transpose(1, 2).view(BATCH, SEQ_LEN, HEADS * HEAD_DIM)
except RuntimeError as err:
    crashed = True
    print(f"original still crashes, as expected:\n  RuntimeError: {err}\n")
assert crashed, "the buggy line should raise — did context get replaced by a contiguous copy?"

for fix in (merge_heads_explicit, merge_heads_reshape):
    merged = fix(context)
    assert merged is not None, f"{fix.__name__} not implemented yet"
    assert merged.shape == (BATCH, SEQ_LEN, HEADS * HEAD_DIM), f"{fix.__name__}: wrong shape {tuple(merged.shape)}"
    assert torch.allclose(merged, reference), f"{fix.__name__}: values scrambled — merged the wrong dims?"
    print(f"{fix.__name__:22s} ✓ matches reference")
print("Exercise 3 passed ✓")

original still crashes, as expected:
  RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

merge_heads_explicit   ✓ matches reference
merge_heads_reshape    ✓ matches reference
Exercise 3 passed ✓


## Exercise 4 — Zero-Copy Mask Broadcast

Attention scores are `(batch, heads, seq_q, seq_k)`; your padding mask is `(batch, seq)` and says which **key** positions are real. Write `broadcast_pad_mask(mask, num_heads)` returning a `(batch, heads, seq, seq)` tensor using **only `unsqueeze` and `expand`** — zero bytes copied.

**Decision you're practicing:** `expand` (free, stride-0, read-only) vs `repeat` (real copy) — and *proving* zero-copy with `.stride()` and storage pointers.

In [8]:
pad_mask = torch.tensor([
    [True, True, True,  False],
    [True, True, False, False],
])                                   # shape: (batch=2, seq=4) — True = real token

def broadcast_pad_mask(mask, num_heads):
    """(batch, seq) -> (batch, num_heads, seq, seq) using ONLY unsqueeze + expand.

    The mask marks KEY positions, so it broadcasts along the query axis:
    every query row sees the same key mask.
    """
    batch_size, seq_len = mask.shape
    key_mask = mask.unsqueeze(1).unsqueeze(2)                       # shape: (batch, 1, 1, seq) — free views
    print(f"after unsqueeze ×2 : shape {tuple(key_mask.shape)}, stride {key_mask.stride()}")
    expanded = key_mask.expand(batch_size, num_heads, seq_len, seq_len)
    print(f"after expand       : shape {tuple(expanded.shape)}, stride {expanded.stride()}")
    print("  → stride 0 on the heads and query dims: those axes re-read the same memory (zero copy)")
    return expanded

NUM_HEADS = 3
attn_mask = broadcast_pad_mask(pad_mask, NUM_HEADS)

after unsqueeze ×2 : shape (2, 1, 1, 4), stride (4, 4, 4, 1)
after expand       : shape (2, 3, 4, 4), stride (4, 0, 0, 1)
  → stride 0 on the heads and query dims: those axes re-read the same memory (zero copy)


**Verification**

In [9]:
# --- Verification: Exercise 4 ---
assert attn_mask is not None, "fill in the stub above first"
assert attn_mask.shape == (2, NUM_HEADS, 4, 4), f"wrong shape: {tuple(attn_mask.shape)}"
assert 0 in attn_mask.stride(), "no stride-0 dims → you copied (repeat?) instead of expanding"
assert attn_mask.untyped_storage().data_ptr() == pad_mask.untyped_storage().data_ptr(), \
    "result must alias the original mask's storage (views only!)"

reference = pad_mask[:, None, None, :].repeat(1, NUM_HEADS, 4, 1)   # the memory-hungry way
assert torch.equal(attn_mask, reference), "values differ from the repeat-based reference"
print(f"zero-copy confirmed: result strides = {attn_mask.stride()}")
print(f"memory check: expand added 0 bytes; repeat would have allocated {reference.numel()} bools")
print("Exercise 4 passed ✓")

zero-copy confirmed: result strides = (4, 0, 0, 1)
memory check: expand added 0 bytes; repeat would have allocated 96 bools
Exercise 4 passed ✓


## Exercise 5 — Broadcast or Bust

For each shape pair, predict the broadcast **result shape** (a tuple) or the string `"error"`. Pair 3 (index starts at 0) is the silent killer from notes §8 — look twice.

**Decision you're practicing:** running the two right-alignment rules in your head *before* PyTorch runs them for you.

In [10]:
shape_pairs = [
    ((4, 1),    (3,)),
    ((2, 3, 4), (3, 4)),
    ((2, 3, 4), (4, 3)),
    ((5,),      (5, 1)),
    ((1,),      (3, 4)),
    ((2, 1, 4), (1, 3, 1)),
    ((3, 2),    (2, 3)),
    ((6, 1, 8), (8,)),
]

predicted_shapes = [
    (4, 3),      # right-align: 1 stretches to 3; missing left dim of (3,) stretches to 4
    (2, 3, 4),   # trailing dims match exactly; missing leading dim stretches to 2
    "error",     # right-aligned pairs are 4 vs 3 and 3 vs 4 — neither side is 1
    (5, 5),      # THE TRAP: (5,) aligns against the trailing 1 → outer-product shape
    (3, 4),      # a lone 1 stretches against everything
    (2, 3, 4),   # the 1s stretch pairwise in both directions
    "error",     # 2 vs 3 mismatch in the last dim
    (6, 1, 8),   # (8,) matches the last dim; the middle 1 has no partner and stays 1
]
for (shape_a, shape_b), prediction in zip(shape_pairs, predicted_shapes):
    print(f"{str(shape_a):12s} ∘ {str(shape_b):12s} → {prediction}")

(4, 1)       ∘ (3,)         → (4, 3)
(2, 3, 4)    ∘ (3, 4)       → (2, 3, 4)
(2, 3, 4)    ∘ (4, 3)       → error
(5,)         ∘ (5, 1)       → (5, 5)
(1,)         ∘ (3, 4)       → (3, 4)
(2, 1, 4)    ∘ (1, 3, 1)    → (2, 3, 4)
(3, 2)       ∘ (2, 3)       → error
(6, 1, 8)    ∘ (8,)         → (6, 1, 8)


**Verification**

In [11]:
# --- Verification: Exercise 5 ---
wrong = 0
for pair_index, (shape_a, shape_b) in enumerate(shape_pairs):
    try:
        actual = tuple(torch.broadcast_shapes(shape_a, shape_b))
    except RuntimeError:
        actual = "error"
    prediction = predicted_shapes[pair_index]
    assert prediction is not None, f"no prediction for pair {pair_index}"
    mark = "✓" if prediction == actual else "✗"
    wrong += prediction != actual
    print(f"{mark} pair {pair_index}: {str(shape_a):12s} vs {str(shape_b):12s} → {str(actual):12s} (you said {prediction})")
if wrong:
    print("\nRe-run the rules: right-align; equal sizes pass; a 1 (or missing dim) stretches; anything else errors.")
assert wrong == 0, f"{wrong} prediction(s) wrong"
print("Exercise 5 passed ✓")

✓ pair 0: (4, 1)       vs (3,)         → (4, 3)       (you said (4, 3))
✓ pair 1: (2, 3, 4)    vs (3, 4)       → (2, 3, 4)    (you said (2, 3, 4))
✓ pair 2: (2, 3, 4)    vs (4, 3)       → error        (you said error)
✓ pair 3: (5,)         vs (5, 1)       → (5, 5)       (you said (5, 5))
✓ pair 4: (1,)         vs (3, 4)       → (3, 4)       (you said (3, 4))
✓ pair 5: (2, 1, 4)    vs (1, 3, 1)    → (2, 3, 4)    (you said (2, 3, 4))
✓ pair 6: (3, 2)       vs (2, 3)       → error        (you said error)
✓ pair 7: (6, 1, 8)    vs (8,)         → (6, 1, 8)    (you said (6, 1, 8))
Exercise 5 passed ✓


## Exercise 6 — Masked Mean Pooling

Turn per-token embeddings `(batch, seq, dim)` into one sentence vector `(batch, dim)` by averaging **only the real tokens** — no Python loops. Row 1 of the test batch has a single real token, so an off-by-`keepdim` bug shows up immediately.

**Decision you're practicing:** broadcasting + reductions with `keepdim=True` — the full shape choreography of notes §8.

In [12]:
def masked_mean_pool(token_embeddings, mask):
    """Average real-token embeddings per sequence.

    Args:
        token_embeddings: float tensor, shape (batch, seq, dim)
        mask: bool tensor, shape (batch, seq) — True where the token is real
    Returns:
        (batch, dim) tensor: per-sequence mean over real tokens only.
    Math note: pooled_b = sum_t(emb_bt * mask_bt) / sum_t(mask_bt)
    """
    mask_float = mask.to(token_embeddings.dtype)             # (batch, seq): bool → float for arithmetic
    masked = token_embeddings * mask_float.unsqueeze(-1)     # (batch, seq, dim): broadcast zeroes the pad rows
    summed = masked.sum(dim=1)                               # (batch, dim): seq is the dim that disappears
    counts = mask_float.sum(dim=1, keepdim=True)             # (batch, 1): keepdim keeps it broadcastable
    pooled = summed / counts                                 # (batch, dim) ÷ (batch, 1) → broadcast divide
    print(f"masked {tuple(masked.shape)} → summed {tuple(summed.shape)} ÷ counts {tuple(counts.shape)} → pooled {tuple(pooled.shape)}")
    return pooled

**Verification**

In [13]:
# --- Verification: Exercise 6 ---
torch.manual_seed(0)
test_embeddings = torch.randn(3, 4, 2)          # shape: (batch=3, seq=4, dim=2)
test_mask = torch.tensor([
    [True, True,  True,  False],
    [True, False, False, False],                # single real token — punishes keepdim mistakes
    [True, True,  False, False],
])

pooled = masked_mean_pool(test_embeddings, test_mask)
assert pooled is not None, "fill in the stub above first"
assert pooled.shape == (3, 2), f"expected (3, 2), got {tuple(pooled.shape)}"

# loop reference: the definition, written the slow obvious way
for row in range(3):
    real_vectors = test_embeddings[row][test_mask[row]]     # (n_real, dim) via boolean indexing
    expected = real_vectors.mean(dim=0)
    assert torch.allclose(pooled[row], expected, atol=1e-6), f"row {row}: {pooled[row]} vs expected {expected}"
    print(f"row {row}: {int(test_mask[row].sum())} real token(s) → pooled {[round(v, 4) for v in pooled[row].tolist()]} ✓")
print("Exercise 6 passed ✓")

masked (3, 4, 2) → summed (3, 2) ÷ counts (3, 1) → pooled (3, 2)
row 0: 3 real token(s) → pooled [-0.1759, -0.2981] ✓
row 1: 1 real token(s) → pooled [0.4681, -0.1577] ✓
row 2: 2 real token(s) → pooled [1.4684, 0.6564] ✓
Exercise 6 passed ✓


## Exercise 7 — dtype Triage

Three tensors, three jobs. Pick the right dtype for each, apply the cast, and record your choice in `dtype_choices`:

1. `big_scores` — raw attention logits reaching 70,000; must **survive a 16-bit cast**
2. `float_token_ids` — token IDs that arrived as floats; must **feed `nn.Embedding`**
3. `tiny_bump` — the value 1.001, where the +0.001 must **survive a 16-bit cast**

**Decision you're practicing:** the dtype table from notes §4 — fp16's *range* problem vs bf16's *precision* problem, and why IDs are `int64`.

In [14]:
big_scores      = torch.tensor([70000.0, -12.5, 300.25])   # shape: (3,)
float_token_ids = torch.tensor([[3.0, 17.0, 0.0]])         # shape: (1, 3)
tiny_bump       = torch.tensor(1.001)                      # scalar

dtype_choices = {
    "big_scores":      torch.bfloat16,   # 70000 > 65504 (fp16 max) → fp16 overflows to inf; bf16 keeps fp32's range
    "float_token_ids": torch.long,       # embedding lookup needs integer row indices — int64 is the convention
    "tiny_bump":       torch.float16,    # fp16 spacing at 1.0 ≈ 0.00098 keeps +0.001; bf16 spacing ≈ 0.0078 rounds it away
}

big_scores_cast      = big_scores.to(dtype_choices["big_scores"])
float_token_ids_cast = float_token_ids.to(dtype_choices["float_token_ids"])
tiny_bump_cast       = tiny_bump.to(dtype_choices["tiny_bump"])

print(f"big_scores  → {dtype_choices['big_scores']}: {big_scores_cast.float().tolist()}")
print(f"              (fp16 would give: {big_scores.to(torch.float16).float().tolist()})")
print(f"token IDs   → {dtype_choices['float_token_ids']}: {float_token_ids_cast.tolist()}")
print(f"tiny_bump   → {dtype_choices['tiny_bump']}: {tiny_bump_cast.item():.10f}")
print(f"              (bf16 would give: {tiny_bump.to(torch.bfloat16).item():.10f} — the bump is gone)")

big_scores  → torch.bfloat16: [70144.0, -12.5, 300.0]
              (fp16 would give: [inf, -12.5, 300.25])
token IDs   → torch.int64: [[3, 17, 0]]
tiny_bump   → torch.float16: 1.0009765625
              (bf16 would give: 1.0000000000 — the bump is gone)


**Verification**

In [15]:
# --- Verification: Exercise 7 ---
assert torch.isinf(big_scores.to(torch.float16)).any(), "sanity: fp16 must overflow on 70000"
assert big_scores_cast is not None, "fill in the stub above first"
assert big_scores_cast.dtype == torch.bfloat16, "big scores need bf16 — fp32 range in 16 bits"
assert torch.isfinite(big_scores_cast.float()).all(), "your cast overflowed!"

assert float_token_ids_cast.dtype == torch.long, "embedding indices must be int64/long"
_ = nn.Embedding(50, 4)(float_token_ids_cast)     # raises on float dtypes
print("nn.Embedding accepted the cast IDs ✓")

assert tiny_bump_cast.dtype == torch.float16, "only fp16 has fine enough spacing at 1.0 to keep +0.001"
assert tiny_bump_cast.item() != 1.0, "the bump vanished — check the 16-bit spacing table in notes §4"
assert tiny_bump.to(torch.bfloat16).item() == 1.0   # sanity: bf16 really does round it away

for key, chosen_dtype in dtype_choices.items():
    print(f"choice: {key:16s} → {chosen_dtype}")
print("Exercise 7 passed ✓")

nn.Embedding accepted the cast IDs ✓
choice: big_scores       → torch.bfloat16
choice: float_token_ids  → torch.int64
choice: tiny_bump        → torch.float16
Exercise 7 passed ✓


---
## Done!

Compare your work against `solutions/ch01-tensors-and-shape-algebra-solution.ipynb`, then move on to **ch02 — Tensor Operations for NLP**. The master decision table at the end of the ch01 notes is worth a re-read now that every row has bitten you at least once.